# Providing access to Google drive

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Loading environment variables

In [3]:
import os
from dotenv import load_dotenv

# The absolute path to your .env file
dotenv_path = '/content/drive/MyDrive/content/.env'

# Load the environment variables
if os.path.exists(dotenv_path):
    load_dotenv(dotenv_path=dotenv_path)
    print("Environment variables loaded successfully.")
else:
    print("Error: .env file not found at the specified path.")

# Access your key
api_key = os.getenv("GEMINI_API_KEY")

Environment variables loaded successfully.


# Defining LLM

In [14]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
from langchain_core.tools import tool
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
    )

### Sample response for a prompt

In [5]:
question1="Which team won the Champions League 2024?"
response1=llm.invoke(question1)
response1.content

'The Champions League final for 2024 has not been played yet!\n\nIt will take place on **June 1, 2024**, at Wembley Stadium in London. The teams competing are:\n\n*   **Real Madrid**\n*   **Borussia Dortmund**'

## Create Dynamic prompts using Prompt Template

In [8]:
from langchain_core.prompts import PromptTemplate
template="Who is the CEO of {company}"
prompt_template=PromptTemplate.from_template(template)
prompt_template

PromptTemplate(input_variables=['company'], input_types={}, partial_variables={}, template='Who is the CEO of {company}')

In [9]:
final_prompt=prompt_template.format(company="Google")
final_prompt

'Who is the CEO of Google'

In [16]:
response=llm.invoke(final_prompt)
response.content

'The CEO of Google (Google LLC) is **Sundar Pichai**.\n\nHe is also the CEO of its parent company, Alphabet Inc.'

## LCEL: LangChain Expression Language
Creating chains that connect prompt to llm to output

In [17]:
from langchain_core.output_parsers import StrOutputParser
output_parser=StrOutputParser()

chain=prompt_template | llm | output_parser
response=chain.invoke({"company": "Google"})
response


"The CEO of Google is **Sundar Pichai**.\n\nHe also serves as the CEO of Google's parent company, Alphabet Inc."

# Document Ingestion

In [ ]:
!pip install langchain-community pypdf langchain-text-splitters


Load PDF content as Markdown

In [7]:
from langchain_community.document_loaders import PyPDFLoader

loader=PyPDFLoader("/content/drive/MyDrive/content/documentation.pdf")
pages = loader.load()
text_content = "\n".join([page.page_content for page in pages])

Define the MarkDownHeaderTextSplitter to decide which headers to split from

In [8]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]

markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on
)
header_splits = markdown_splitter.split_text(text_content)

Next, split those sections into smaller chunks if they are too big

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)
final_chunks = text_splitter.split_documents(header_splits)

Check the result

In [ ]:
for chunk in final_chunks[:3]:
    print(f"Metadata: {chunk.metadata}")
    print(f"Content Snippet: {chunk.page_content[:100]}...")
    print("-" * 20)

## Embedding and Storing Chunks

In [ ]:
!pip install -U langchain-chroma langchain-google-genai

Saving chunks in drive

In [10]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
import os

# Initialize the Embedding Model
embeddings = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")

# Create the Vector Database
# 'persist_directory' saves the data so it doesn't disappear when the cell finishes
vector_db = Chroma.from_documents(
    documents=final_chunks,
    embedding=embeddings,
    persist_directory="./architecture_db"
)

print("Vector database created and saved successfully!")

Vector database created and saved successfully!


## LCEL Retrieval Chain

Creating a prompt template

In [11]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Create the Prompt Template
# We tell the AI to ONLY use the provided context to answer.
template = """
You are an expert Software Architect. Use the following pieces of retrieved context
from the architecture documentation to answer the user's question.

If you don't know the answer based on the context, just say you don't know.
Keep the answer concise and professional.

Context:
{context}

Question:
{question}

Answer:
"""
prompt = ChatPromptTemplate.from_template(template)

Defining a Retriever

In [18]:
# This turns our database into a tool that fetches the top 3 relevant chunks
retriever = vector_db.as_retriever(search_kwargs={"k": 8})

Build the RAG chain and run it

In [ ]:
# The 'RunnablePassthrough' allows us to pass the question to both the retriever and the prompt
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# Running with the final prompt and getting the output response

In [20]:
question = "What are the features in Scenario section defined under each core quality attributes mentioned in the introduction?"
response = rag_chain.invoke(question)

print(f"QUESTION: {question}")
print("-" * 30)
print(f"AI ARCHITECT: {response}")

QUESTION: What are the features in Scenario section defined under each core quality attributes mentioned in the introduction?
------------------------------
AI ARCHITECT: The core quality attributes mentioned in the introduction are Usability, Availability, Maintainability, and Testability. The scenarios defined under each are:

**Usability:**
*   The end user wants to discover what features are available to them.
*   The end user wants quick access to core features for their user class to improve efficiency of use.
*   The user wants to receive user and situation appropriate error messages when an error occurs.

**Availability:**
*   The system times out before a Work Report or Employer Evaluation can be submitted.
*   The application server fails or becomes unresponsive, causing the entire system to fail.
*   The system is compromised by a Denial of Service (DoS) attack.

**Maintainability:**
*   Codebase is large and complex making it difficult to add new features.
*   Lack of docum